# Phase 2 Baseline + Hybrid (Colab Ready)

This notebook runs 4 evaluation tracks on `test.jsonl`:
1. `Llama-ZeroShot` (Groq)
2. `Llama-Prompted` (Groq)
3. `Mistral-Prompted` (HF Inference API)
4. `Hybrid` = Llama for strict structured fields + Mistral for description text

Outputs are saved to `/content/phase2_hybrid_outputs`.


In [ ]:
!pip install -q groq huggingface_hub scikit-learn rouge-score nltk pandas requests


In [ ]:
import os
import json
import re
import time
import glob
from pathlib import Path

import requests
import pandas as pd
import nltk
from groq import Groq
from huggingface_hub import InferenceClient
from sklearn.metrics import accuracy_score, f1_score, classification_report
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
print('Imports ready')


In [ ]:
# Paths (same style as your previous Colab notebook)
candidates = ['/content/test.jsonl', '/test.jsonl']
test_matches = [p for p in candidates if Path(p).exists()]
if not test_matches:
    test_matches = glob.glob('/kaggle/input/**/test.jsonl', recursive=True)

if not test_matches:
    raise FileNotFoundError('test.jsonl not found in /content or kaggle input')

TEST_FILE = test_matches[0]
DATA_DIR = str(Path(TEST_FILE).parent)
TRAIN_FILE = str(Path(DATA_DIR) / 'train.jsonl')
VAL_FILE = str(Path(DATA_DIR) / 'val.jsonl')
MASTER_FILE = str(Path(DATA_DIR) / 'master_with_splits.json')

if not Path(TRAIN_FILE).exists() and Path('/content/train.jsonl').exists():
    TRAIN_FILE = '/content/train.jsonl'
if not Path(VAL_FILE).exists() and Path('/content/val.jsonl').exists():
    VAL_FILE = '/content/val.jsonl'
if not Path(MASTER_FILE).exists() and Path('/content/master_with_splits.json').exists():
    MASTER_FILE = '/content/master_with_splits.json'
if not Path(MASTER_FILE).exists():
    MASTER_FILE = None

OUTPUT_DIR = '/content/phase2_hybrid_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('TEST_FILE  :', TEST_FILE)
print('TRAIN_FILE :', TRAIN_FILE, '| exists:', Path(TRAIN_FILE).exists())
print('VAL_FILE   :', VAL_FILE,   '| exists:', Path(VAL_FILE).exists())
print('MASTER_FILE:', MASTER_FILE)
print('OUTPUT_DIR :', OUTPUT_DIR)


In [ ]:
# Load secrets (Colab first, then env fallback)
# Required secrets:
# - HF_TOKEN
# - GROQ_API or GROQ_API_KEY

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
    try:
        os.environ['GROQ_API'] = sec.get_secret('GROQ_API')
    except Exception:
        os.environ['GROQ_API_KEY'] = sec.get_secret('GROQ_API_KEY')
else:
    try:
        from google.colab import userdata
        hf = userdata.get('HF_TOKEN')
        gq = None
        try:
            gq = userdata.get('GROQ_API')
        except Exception:
            gq = None
        if not gq:
            try:
                gq = userdata.get('GROQ_API_KEY')
            except Exception:
                gq = None

        if hf:
            os.environ['HF_TOKEN'] = hf
        if gq:
            os.environ['GROQ_API'] = gq
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    raise EnvironmentError('HF_TOKEN not set')

GROQ_KEY = os.environ.get('GROQ_API') or os.environ.get('GROQ_API_KEY')
if not GROQ_KEY:
    raise EnvironmentError('GROQ_API (or GROQ_API_KEY) not set')

print('HF_TOKEN loaded  :', bool(os.environ.get('HF_TOKEN')))
print('GROQ_KEY loaded  :', bool(GROQ_KEY))


In [ ]:
# Model config
# T4 compatibility: this notebook uses API calls; local GPU VRAM is not the bottleneck.

LLAMA_MODEL = 'llama-3.3-70b-versatile'                  # Groq
MISTRAL_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2:featherless-ai'  # HF routed provider

MAX_RETRIES = 3
RETRY_BASE_SEC = 1.5
DELAY_SEC = 1.0

groq_client = Groq(api_key=GROQ_KEY)
hf_client = InferenceClient(token=os.environ['HF_TOKEN'], timeout=120)

print('LLAMA_MODEL  :', LLAMA_MODEL)
print('MISTRAL_MODEL:', MISTRAL_MODEL)


In [ ]:
# Data loading + dynamic label sets

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_master_records(master_path, fallback_rows):
    if not master_path:
        return fallback_rows
    with open(master_path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        if 'records' in obj and isinstance(obj['records'], list):
            return obj['records']
        if 'data' in obj and isinstance(obj['data'], list):
            return obj['data']
    return fallback_rows


test_records = load_jsonl(TEST_FILE)
master_records = load_master_records(MASTER_FILE, test_records)

VALID_CATEGORIES = sorted({str(r.get('category', '')).strip() for r in master_records if r.get('category')})
VALID_SEVERITIES = sorted({str(r.get('severity', '')).strip().lower() for r in master_records if r.get('severity')})
VALID_DEPARTMENTS = sorted({
    str(r.get('routing', {}).get('primary_department', '')).strip()
    for r in master_records if r.get('routing', {}).get('primary_department')
})

print(f'Loaded {len(test_records)} test records')
print('VALID_CATEGORIES :', VALID_CATEGORIES)
print('VALID_SEVERITIES :', VALID_SEVERITIES)
print('VALID_DEPARTMENTS:', VALID_DEPARTMENTS)


In [ ]:
# Prompt builders

def build_zeroshot_prompt(caption, voice_text):
    return f"""A patient filed a hospital complaint.
Image caption: {caption}
Voice complaint: {voice_text}

Return JSON with keys:
category, severity, department, complaint_description"""


def build_prompted_prompt(caption, voice_text):
    cat_str = "\\n".join(f"- {c}" for c in VALID_CATEGORIES)
    dep_str = ", ".join(VALID_DEPARTMENTS)

    return f"""You are a hospital complaint triage system.
Return ONLY valid JSON (no markdown) with keys:
category, severity, department, complaint_description

Use exactly one category from:
{cat_str}

Use severity from:
- low
- medium
- high
- critical

Use department from:
{dep_str}

Examples:
Input: bathroom mold + patient says bathroom filthy
Output: {{"category":"Dirty Hospital Bathroom","severity":"high","department":"Housekeeping","complaint_description":"Patient reported severe unhygienic bathroom conditions with visible mold and foul odor."}}

Input: rat near kitchen + patient says rat seen near food area
Output: {{"category":"Rats / Rodent Infestation","severity":"critical","department":"Pest Control","complaint_description":"Patient observed rodent presence near food handling area, creating major hygiene risk."}}

Now process:
Image caption: {caption}
Voice complaint: {voice_text}
"""


def build_mistral_description_prompt(caption, voice_text, category, severity, department):
    return f"""You write only complaint description text for a hospital triage JSON.
Do not output JSON. Output only 1-2 sentences.
Stay grounded in given inputs and do not invent facts.

Image caption: {caption}
Voice complaint: {voice_text}
Chosen category: {category}
Chosen severity: {severity}
Chosen department: {department}

Write the complaint description now:"""


In [ ]:
# Parsing helpers

def parse_json_output(raw_text):
    if not raw_text:
        return {}, False

    cleaned = re.sub(r'```(?:json)?', '', str(raw_text), flags=re.IGNORECASE).replace('```', '').strip()
    s = cleaned.find('{')
    e = cleaned.rfind('}')
    if s != -1 and e != -1 and e > s:
        block = cleaned[s:e+1]
        try:
            return json.loads(block), True
        except Exception:
            pass

    try:
        return json.loads(cleaned), True
    except Exception:
        return {}, False


def canonicalize(value, allowed, lower=False):
    if value is None:
        return ''
    raw = str(value).strip()
    if lower:
        raw_cmp = raw.lower()
        lut = {x.lower(): x for x in allowed}
    else:
        raw_cmp = raw
        lut = {x: x for x in allowed}
    return lut.get(raw_cmp, raw.lower() if lower else raw)


def extract_fields(parsed):
    cat = canonicalize(parsed.get('category', ''), VALID_CATEGORIES, lower=False)
    sev = canonicalize(parsed.get('severity', ''), VALID_SEVERITIES, lower=True)
    dep = canonicalize(parsed.get('department', ''), VALID_DEPARTMENTS, lower=False)
    desc = str(parsed.get('complaint_description', '')).strip()
    return cat, sev, dep, desc


In [ ]:
# API callers

def call_groq(prompt, record_id, track_name):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = groq_client.chat.completions.create(
                model=LLAMA_MODEL,
                messages=[
                    {'role': 'system', 'content': 'Return only valid JSON when asked for JSON.'},
                    {'role': 'user', 'content': prompt}
                ],
                temperature=0,
                max_tokens=512,
            )
            return (resp.choices[0].message.content or '').strip()
        except Exception as e:
            last_err = str(e)
            retryable = any(k in last_err.lower() for k in ['429', 'rate', 'timeout', '503', '502'])
            if retryable and attempt < MAX_RETRIES:
                time.sleep(RETRY_BASE_SEC * (2 ** (attempt - 1)))
                continue
            break
    print(f'  ERROR on {record_id} [{track_name}-Groq]: {last_err}')
    return None


def call_hf_mistral_chat(prompt, record_id, track_name):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if hasattr(hf_client, 'chat') and hasattr(hf_client.chat, 'completions'):
                resp = hf_client.chat.completions.create(
                    model=MISTRAL_MODEL,
                    messages=[
                        {'role': 'system', 'content': 'Follow user instruction exactly.'},
                        {'role': 'user', 'content': prompt},
                    ],
                    temperature=0,
                    max_tokens=512,
                )
            else:
                resp = hf_client.chat_completion(
                    model=MISTRAL_MODEL,
                    messages=[
                        {'role': 'system', 'content': 'Follow user instruction exactly.'},
                        {'role': 'user', 'content': prompt},
                    ],
                    temperature=0,
                    max_tokens=512,
                )
            content = resp.choices[0].message.content
            if isinstance(content, str):
                return content.strip()
            if isinstance(content, list):
                return ''.join((x.get('text', '') if isinstance(x, dict) else str(x)) for x in content).strip()
            return str(content).strip()
        except Exception as e:
            last_err = str(e)
            retryable = any(k in last_err.lower() for k in ['429', 'rate', 'timeout', '503', '502'])
            if retryable and attempt < MAX_RETRIES:
                time.sleep(RETRY_BASE_SEC * (2 ** (attempt - 1)))
                continue
            break
    print(f'  ERROR on {record_id} [{track_name}-HF]: {last_err}')
    return None


In [ ]:
# Metrics + runners

def compute_metrics(results, track_name, model_name):
    gt_c = [r['gt_category'] for r in results]
    gt_s = [r['gt_severity'] for r in results]
    gt_d = [r['gt_department'] for r in results]

    pr_c = [r['pred_category'] for r in results]
    pr_s = [r['pred_severity'] for r in results]
    pr_d = [r['pred_department'] for r in results]

    json_valid_count = sum(1 for r in results if r['json_valid'])

    cat_acc = accuracy_score(gt_c, pr_c)
    sev_acc = accuracy_score(gt_s, pr_s)
    dep_acc = accuracy_score(gt_d, pr_d)

    cat_f1 = f1_score(gt_c, pr_c, average='macro', zero_division=0)
    sev_f1 = f1_score(gt_s, pr_s, average='macro', zero_division=0)
    dep_f1 = f1_score(gt_d, pr_d, average='macro', zero_division=0)

    invalid_cat = sum(1 for x in pr_c if x not in VALID_CATEGORIES)
    invalid_sev = sum(1 for x in pr_s if x not in VALID_SEVERITIES)
    invalid_dep = sum(1 for x in pr_d if x not in VALID_DEPARTMENTS)

    smoother = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    bleu, r1, r2, rl = [], [], [], []
    for r in results:
        ref = r['gt_description'].split()
        hyp = r['pred_description'].split()
        if ref and hyp:
            bleu.append(sentence_bleu([ref], hyp, smoothing_function=smoother))
            rs = scorer.score(r['gt_description'], r['pred_description'])
            r1.append(rs['rouge1'].fmeasure)
            r2.append(rs['rouge2'].fmeasure)
            rl.append(rs['rougeL'].fmeasure)

    def avg(vals):
        return round(sum(vals) / len(vals), 4) if vals else 0.0

    print(f"\\nPer-class CATEGORY report [{track_name}]")
    print(classification_report(gt_c, pr_c, zero_division=0))

    return {
        'baseline': track_name,
        'model': model_name,
        'total_records': len(results),
        'json_validity_rate': round(json_valid_count / len(results), 4),
        'category_accuracy': round(cat_acc, 4),
        'category_macro_f1': round(cat_f1, 4),
        'severity_accuracy': round(sev_acc, 4),
        'severity_macro_f1': round(sev_f1, 4),
        'department_accuracy': round(dep_acc, 4),
        'department_macro_f1': round(dep_f1, 4),
        'bleu_score': avg(bleu),
        'rouge1': avg(r1),
        'rouge2': avg(r2),
        'rougeL': avg(rl),
        'invalid_category_preds': invalid_cat,
        'invalid_severity_preds': invalid_sev,
        'invalid_department_preds': invalid_dep,
    }


def run_track(rows, track_name, json_prompt_builder, json_caller, hybrid_desc=False):
    print("\\n" + '=' * 66)
    print(f"Running: {track_name} ({len(rows)} records)")
    print('=' * 66)

    out = []
    ckpt = os.path.join(OUTPUT_DIR, f"checkpoint_{track_name.replace(' ', '_')}.csv")

    for i, rec in enumerate(rows, 1):
        image_id = rec.get('image_id', f'row_{i}')
        caption = rec.get('input', {}).get('image_caption', rec.get('refined_caption', ''))
        voice = rec.get('voice_text', '')

        gt_cat = str(rec.get('category', '')).strip()
        gt_sev = str(rec.get('severity', '')).strip().lower()
        gt_dep = str(rec.get('routing', {}).get('primary_department', rec.get('department', ''))).strip()
        gt_desc = str(rec.get('complaint_description', '')).strip()

        print(f"  [{i:02d}/{len(rows)}] {image_id} ... ", end='', flush=True)

        base_prompt = json_prompt_builder(caption, voice)
        raw = json_caller(base_prompt, image_id, track_name)
        parsed, is_valid = parse_json_output(raw)
        pr_cat, pr_sev, pr_dep, pr_desc = extract_fields(parsed)

        if hybrid_desc:
            desc_prompt = build_mistral_description_prompt(
                caption=caption,
                voice_text=voice,
                category=pr_cat or gt_cat,
                severity=pr_sev or gt_sev,
                department=pr_dep or gt_dep,
            )
            desc_text = call_hf_mistral_chat(desc_prompt, image_id, track_name)
            if desc_text:
                desc_text = re.sub(r'^```.*?\\n', '', desc_text, flags=re.DOTALL).replace('```', '').strip()
                d_parsed, d_valid = parse_json_output(desc_text)
                if d_valid and isinstance(d_parsed, dict) and d_parsed.get('complaint_description'):
                    pr_desc = str(d_parsed.get('complaint_description')).strip()
                else:
                    pr_desc = desc_text

        cat_ok = (pr_cat == gt_cat)
        sev_ok = (pr_sev == gt_sev)
        dep_ok = (pr_dep == gt_dep)

        print(('cat✓' if cat_ok else 'cat✗'), '|', ('sev✓' if sev_ok else 'sev✗'), '|', ('json✓' if is_valid else 'json✗'))

        out.append({
            'baseline': track_name,
            'image_id': image_id,
            'gt_category': gt_cat,
            'gt_severity': gt_sev,
            'gt_department': gt_dep,
            'gt_description': gt_desc,
            'pred_category': pr_cat,
            'pred_severity': pr_sev,
            'pred_department': pr_dep,
            'pred_description': pr_desc,
            'json_valid': is_valid,
            'raw_output': raw or '',
            'cat_correct': cat_ok,
            'sev_correct': sev_ok,
            'dept_correct': dep_ok,
        })

        pd.DataFrame(out).to_csv(ckpt, index=False)
        time.sleep(DELAY_SEC)

    return out


In [ ]:
# Smoke tests
print('Smoke test Llama:')
smoke_l = call_groq(
    build_zeroshot_prompt('A broken hospital bed with unsafe rail', 'My bed is broken and unsafe'),
    'smoke_llama',
    'Smoke',
)
print(smoke_l)

print('\\nSmoke test Mistral:')
smoke_m = call_hf_mistral_chat(
    build_prompted_prompt('A broken hospital bed with unsafe rail', 'My bed is broken and unsafe'),
    'smoke_mistral',
    'Smoke',
)
print(smoke_m)


In [ ]:
# Run all tracks
results_llama_zs = run_track(test_records, 'Llama-ZeroShot', build_zeroshot_prompt, call_groq, hybrid_desc=False)
metrics_llama_zs = compute_metrics(results_llama_zs, 'Llama-ZeroShot', LLAMA_MODEL)

results_llama_pt = run_track(test_records, 'Llama-Prompted', build_prompted_prompt, call_groq, hybrid_desc=False)
metrics_llama_pt = compute_metrics(results_llama_pt, 'Llama-Prompted', LLAMA_MODEL)

results_mistral_pt = run_track(test_records, 'Mistral-Prompted', build_prompted_prompt, call_hf_mistral_chat, hybrid_desc=False)
metrics_mistral_pt = compute_metrics(results_mistral_pt, 'Mistral-Prompted', MISTRAL_MODEL)

results_hybrid = run_track(test_records, 'Hybrid-LlamaStruct-MistralDesc', build_prompted_prompt, call_groq, hybrid_desc=True)
metrics_hybrid = compute_metrics(
    results_hybrid,
    'Hybrid-LlamaStruct-MistralDesc',
    f'{LLAMA_MODEL} + {MISTRAL_MODEL}',
)

all_results = results_llama_zs + results_llama_pt + results_mistral_pt + results_hybrid
all_metrics = [metrics_llama_zs, metrics_llama_pt, metrics_mistral_pt, metrics_hybrid]

results_csv = os.path.join(OUTPUT_DIR, 'baseline_results.csv')
metrics_json = os.path.join(OUTPUT_DIR, 'baseline_metrics.json')
errors_csv = os.path.join(OUTPUT_DIR, 'baseline_errors.csv')
summary_txt = os.path.join(OUTPUT_DIR, 'run_summary.txt')

pd.DataFrame(all_results).to_csv(results_csv, index=False)
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

errors = [r for r in all_results if (not r['cat_correct']) or (not r['sev_correct']) or (not r['json_valid'])]
pd.DataFrame(errors).to_csv(errors_csv, index=False)

summary_cols = [
    'baseline','model','json_validity_rate',
    'category_accuracy','category_macro_f1',
    'severity_accuracy','severity_macro_f1',
    'department_accuracy','department_macro_f1',
    'bleu_score','rouge1','rouge2','rougeL',
]
summary_df = pd.DataFrame(all_metrics)[summary_cols]

with open(summary_txt, 'w', encoding='utf-8') as f:
    f.write(summary_df.to_string(index=False))

print('\\nSaved files:')
print(' -', results_csv)
print(' -', metrics_json)
print(' -', errors_csv)
print(' -', summary_txt)

print('\\nSummary table:')
print(summary_df)


In [ ]:
# Download outputs as ZIP (Colab)
import zipfile
from google.colab import files

zip_path = '/content/phase2_hybrid_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, arcname=fn)

print('Created:', zip_path)
files.download(zip_path)
